# Relay Trace To RL Dataset

This notebook runs the local Relay-instrumented agent, captures ATOF JSONL, and creates two different datasets:

- structural RL episodes/transitions for trajectory scoring
- faithful model-call samples that preserve the model request and assistant target for token-level learning


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
TRACE = ROOT / "outputs" / "managed_custom_trace_events.jsonl"
RL_DIR = ROOT / "outputs" / "rl"
MODEL_CALL_DIR = ROOT / "outputs" / "model_calls"
print(ROOT)


Run the offline managed custom-StateGraph demo. This writes the Relay trace to `outputs/managed_custom_trace_events.jsonl`.

In [ ]:
! .venv/bin/agent-managed-custom-demo

Convert the Relay trace into RL-style episode and transition JSONL.

In [ ]:
! .venv/bin/relay-trace-rl-pipeline --input outputs/managed_custom_trace_events.jsonl --output-dir outputs/rl

In [ ]:
print((RL_DIR / "summary.md").read_text())

In [ ]:
import json

with (RL_DIR / "transitions.jsonl").open() as f:
    first_transition = json.loads(next(f))
first_transition

## Faithful Model-Call Samples

The RL transition records above intentionally describe structure and reward. They do not teach a model which assistant tokens or tool-call arguments to generate.

The next step extracts each successful Relay `llm` scope as a supervised sample: request `messages` plus the target assistant message. Tool calls are preserved as structured function calls, and follow-up turns keep the `tool_call_id` link between the assistant tool call and the tool response.


In [ ]:
! .venv/bin/relay-model-call-samples --trace outputs/managed_custom_trace_events.jsonl --output-dir outputs/model_calls


In [ ]:
print((MODEL_CALL_DIR / "summary.md").read_text())


In [ ]:
with (MODEL_CALL_DIR / "model_call_samples.jsonl").open() as f:
    model_call_samples = [json.loads(line) for line in f]

[(sample["scope_name"], sample["target_kind"], sample["target"]) for sample in model_call_samples]


In [ ]:
with (MODEL_CALL_DIR / "sft_messages.jsonl").open() as f:
    first_content_sample = next(
        json.loads(line)
        for line in f
        if json.loads(line)["metadata"]["target_kind"] == "content"
    )

first_content_sample
